# RAG MVP (recetas)

Este cuaderno orquesta el pipeline usando los modulos en src/.

In [1]:
# En Colab, instala dependencias (si ya estan, puedes saltar esta celda).
!pip -q install -r ../requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'


In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.chunking import chunk_documents
from src.data_loader import build_documents, load_recipes
from src.embeddings import embed_passages, load_embedder
from src.faiss_index import build_index, save_docs, save_index

DATA_PATH = ROOT / "ChefGPT_Dataset_Random_Sample.json"
OUT_DIR = ROOT / "artifacts"

recipes = load_recipes(DATA_PATH)
docs = build_documents(recipes)
chunks = chunk_documents(docs, max_words=220, overlap=40)

texts = [doc["text"] for doc in chunks]
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = load_embedder("intfloat/multilingual-e5-base", device=device)
embeddings = embed_passages(embedder, texts, batch_size=32)

index = build_index(embeddings)
save_index(index, OUT_DIR / "index.faiss")
save_docs(chunks, OUT_DIR / "docs.jsonl")

print(f"Index listo: {len(chunks)} chunks")

In [ ]:
from src.embeddings import embed_query
from src.faiss_index import load_docs, load_index
from src.rag_pipeline import generate_answer, load_llm

index = load_index(OUT_DIR / "index.faiss")
docs = load_docs(OUT_DIR / "docs.jsonl")

query = "Tengo pollo y arroz. Que puedo cocinar rapido?"
query_vec = embed_query(embedder, query).astype("float32")
scores, indices = index.search(query_vec.reshape(1, -1), 6)
contexts = [docs[i]["text"] for i in indices[0] if i >= 0]

tokenizer, model = load_llm("Qwen/Qwen2.5-7B-Instruct")
answer = generate_answer(tokenizer, model, query, contexts)
print(answer)